# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Davionnic/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Paper: *The State of AI-Driven SEO in Numbers* (FlyRank, March 2026). Constructive tone — how to make the claim stronger, not a gotcha.

### Finding A — Finding #4 "The Freshness Multiplier"
**Claim (paper):** 365+ day content refreshed within 30 days shows a **3.2× health** lift (10.7 → 34.5) and **57× impressions** (71 → 4039); refresh timing is framed as one of the strongest measured levers.

| Question | Answer |
|---|---|
| Where does the label / outcome come from? | **FlyRank health score** (composite of impressions, position, CTR, scroll) **and** raw impressions — observational cohorts (refreshed mature pages vs others), not a randomized or matched experiment. |
| Does the validation design carry the claim? | **Partly for "associated with," not for "refresh produces / causes."** Chosen pages may already be higher-potential (selection). Health embeds the same visibility inputs the lift is judged on (circular flavor). Tiny / unstable tails are acknowledged elsewhere (e.g. 361+ growth ratio), which is good — the action copy ("produces 3.2×") still overreaches. |
| Stronger next step | Pre-register comparable untreated mature pages; report pre/post with matching; lead with impressions/clicks, treat health as context only. |

### Finding B — ML appendix "What Predicts Health?" (Random Forest)
**Claim (paper):** Average Position ~43% importance, Impressions ~32%, Scroll ~15% when an RF predicts health score (holdout-tested).

| Question | Answer |
|---|---|
| Where does the label come from? | **Health score** — the paper itself says it is partly **constructed from** position, impressions, CTR, scroll. |
| Does the validation design carry the claim? | **Holdout tests generalization of a circular target**, not discovery of external SEO drivers. The paper *discloses* this (descriptive, not causal) — that honesty is the right bar. A reader can still misread the chart as "optimize position to raise health." |
| Stronger next step | Keep it labeled "reconstructing a known composite"; or predict a **future** outcome (e.g. later impression drop) with features frozen before the label window. |

**Takeaway for my lane:** Same discipline applies to my ML-08 ranker — Precision@K is decision-support on a same-window proxy, not proof that refreshing the top of the queue recovers traffic.


In [1]:
import os
from pathlib import Path

# walk up until we find the repo root (works from work/notebooks/ or Colab)
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

PAPER = Path("docs/flyrank-seo-research-march-2026.pdf")
assert PAPER.exists(), f"missing paper at {PAPER.resolve()}"
print("Working dir:", os.getcwd())
print(f"paper present: {PAPER} ({PAPER.stat().st_size:,} bytes)")
print("audited findings: Finding #4 Freshness Multiplier + ML appendix RF→health_score")
print("claim ladder in use: observed / measured / directional / decision-support (not causal)")
print("lane context: Refresh / Content Opportunity Scoring (same as ML-08)")


Working dir: /workspace/flyrank-ml-internship
paper present: docs/flyrank-seo-research-march-2026.pdf (199,961 bytes)
audited findings: Finding #4 Freshness Multiplier + ML appendix RF→health_score
claim ladder in use: observed / measured / directional / decision-support (not causal)
lane context: Refresh / Content Opportunity Scoring (same as ML-08)


## 2. My model under an honest split (before / after)

**Model:** Week-5 / ML-08 best method — **random_forest** (`max_depth=10`, 200 trees, `balanced_subsample`), features = `MODEL_NUMERIC_FEATURES` + one-hot categoricals. Label = `is_declining_label = (trend_direction == "down")`. Primary metric = **Precision@50**.

**Before (weaker split):** random stratified **row** holdout at the same ~7.75% test fraction.

**After (honest split):** **client-holdout** (~20% of clients) — same design as ML-07 / ML-08 / `scripts/03_train_model.py` (`RANDOM_STATE=42`). Time-aware split is not available on this starter CSV (same-window teaching slice; no future outcome window).

**Measured on this run:**

| Split | Precision@50 | Avg precision | ROC-AUC | Holdout base rate | Test rows |
|---|---:|---:|---:|---:|---:|
| Random row (before) | **0.900** | 0.776 | 0.771 | 0.542 | 2,325 |
| Client-holdout (after) | **0.680** | 0.610 | 0.747 | 0.391 | 2,325 |

**Gap = 0.220 Precision@50** (0.900 → 0.680). Random rows let the model borrow client "style" across folds and sit on a easier base rate; client-holdout asks whether the score helps on accounts never seen in training — that is the number that matches the product decision. ML-08's reported **0.680** is the honest one; the random **0.900** is the memorization-inflated twin.


In [2]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
import sklearn

sys.path.insert(0, str(Path("scripts").resolve()))
from ml_utils import (  # noqa: E402
    MODEL_CATEGORICAL_FEATURES,
    MODEL_NUMERIC_FEATURES,
    precision_at_k,
)

RAW = "data/raw/content_refresh_anonymized.csv"
FEAT = "data/processed/refresh_feature_vector.csv"
assert os.path.exists(RAW), f"missing starter CSV at {os.path.abspath(RAW)}"

if not os.path.exists(FEAT):
    import runpy

    runpy.run_path("scripts/01_prepare_features.py", run_name="__main__")

df = pd.read_csv(FEAT)
assert "is_declining_label" in df.columns
assert (
    df["is_declining_label"]
    == (df["trend_direction"].astype(str).str.lower() == "down").astype(int)
).all()

RANDOM_STATE = 42


def build_feature_matrix(frame: pd.DataFrame, extra_cols: list[str] | None = None):
    numeric_features = [c for c in MODEL_NUMERIC_FEATURES if c in frame.columns]
    categorical_features = [c for c in MODEL_CATEGORICAL_FEATURES if c in frame.columns]

    numeric_frame = frame[numeric_features].apply(pd.to_numeric, errors="coerce")
    numeric_frame = numeric_frame.replace([np.inf, -np.inf], np.nan).fillna(0)

    categorical_frame = frame[categorical_features].fillna("unknown").astype(str)
    encoded_frame = pd.get_dummies(
        categorical_frame,
        prefix=categorical_features,
        dummy_na=False,
        dtype=float,
    )

    feature_frame = pd.concat(
        [numeric_frame.reset_index(drop=True), encoded_frame.reset_index(drop=True)],
        axis=1,
    )
    if extra_cols:
        for col in extra_cols:
            if col == "trend_direction":
                feature_frame["leak_trend_down"] = (
                    frame["trend_direction"].astype(str).str.lower() == "down"
                ).astype(float).to_numpy()
            else:
                feature_frame[f"leak_{col}"] = (
                    pd.to_numeric(frame[col], errors="coerce").fillna(0).to_numpy()
                )
    forbidden = {"trend_direction", "trend_pct", "is_declining_label"}
    overlap = forbidden.intersection(feature_frame.columns)
    assert not overlap, f"LEAK into feature matrix: {overlap}"
    return feature_frame, list(feature_frame.columns)


def make_client_holdout(frame: pd.DataFrame):
    all_indices = np.arange(len(frame))
    client_series = frame["client_id"].fillna("unknown").astype(str)
    unique_clients = client_series.drop_duplicates().to_numpy()
    rng = np.random.default_rng(RANDOM_STATE)
    shuffled = rng.permutation(unique_clients)
    test_client_count = max(1, int(round(len(shuffled) * 0.2)))
    test_clients = set(shuffled[:test_client_count].tolist())
    test_mask = client_series.isin(test_clients).to_numpy()
    return all_indices[~test_mask], all_indices[test_mask], test_clients


def make_rf() -> RandomForestClassifier:
    return RandomForestClassifier(
        class_weight="balanced_subsample",
        max_depth=10,
        min_samples_leaf=25,
        n_estimators=200,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )


def ranking_metrics(y_true: pd.Series, scores: np.ndarray) -> dict[str, float]:
    return {
        "precision_at_20": precision_at_k(y_true, scores, 20),
        "precision_at_50": precision_at_k(y_true, scores, 50),
        "average_precision": float(average_precision_score(y_true, scores)),
        "roc_auc": float(roc_auc_score(y_true, scores)),
    }


y = df["is_declining_label"].astype(int)
X, feature_columns = build_feature_matrix(df)

train_idx, test_idx, test_clients = make_client_holdout(df)
test_frac = len(test_idx) / len(df)

# --- AFTER: honest client-holdout (matches ML-08) ---
rf_client = make_rf()
rf_client.fit(X.iloc[train_idx], y.iloc[train_idx])
scores_client = rf_client.predict_proba(X.iloc[test_idx])[:, 1]
metrics_client = ranking_metrics(y.iloc[test_idx], scores_client)
base_client = float(y.iloc[test_idx].mean())

# --- BEFORE: random stratified row split (same test size) ---
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(
    X, y, test_size=test_frac, random_state=RANDOM_STATE, stratify=y
)
rf_random = make_rf()
rf_random.fit(Xtr_r, ytr_r)
scores_random = rf_random.predict_proba(Xte_r)[:, 1]
metrics_random = ranking_metrics(yte_r, scores_random)
base_random = float(yte_r.mean())

gap_p50 = metrics_random["precision_at_50"] - metrics_client["precision_at_50"]

print("=== honest split audit (Week-5 random_forest) ===")
print(f"sklearn={sklearn.__version__}  RANDOM_STATE={RANDOM_STATE}")
print(f"features={len(feature_columns)}  clients={df['client_id'].nunique()}")
print(f"test_frac≈{test_frac:.4f}  test_clients={sorted(test_clients)}")
print()
print("--- BEFORE: random stratified row split ---")
print(
    f"Precision@50={metrics_random['precision_at_50']:.3f}  "
    f"AP={metrics_random['average_precision']:.3f}  "
    f"ROC-AUC={metrics_random['roc_auc']:.3f}  "
    f"base_rate={base_random:.3f}  n={len(yte_r):,}"
)
print("--- AFTER: client-holdout (honest) ---")
print(
    f"Precision@50={metrics_client['precision_at_50']:.3f}  "
    f"AP={metrics_client['average_precision']:.3f}  "
    f"ROC-AUC={metrics_client['roc_auc']:.3f}  "
    f"base_rate={base_client:.3f}  n={len(test_idx):,}"
)
print(f"GAP (random − client) Precision@50 = {gap_p50:.3f}")
print("time-aware split: N/A on starter same-window CSV (disclosed limitation)")


=== honest split audit (Week-5 random_forest) ===
sklearn=1.9.1  RANDOM_STATE=42
features=52  clients=32
test_frac≈0.0775  test_clients=['client_0b918943df', 'client_1a6562590e', 'client_4fc82b26ae', 'client_98a3ab7c34', 'client_d4735e3a26', 'client_f74efabef1']

--- BEFORE: random stratified row split ---
Precision@50=0.900  AP=0.776  ROC-AUC=0.771  base_rate=0.542  n=2,325
--- AFTER: client-holdout (honest) ---
Precision@50=0.680  AP=0.610  ROC-AUC=0.747  base_rate=0.391  n=2,325
GAP (random − client) Precision@50 = 0.220
time-aware split: N/A on starter same-window CSV (disclosed limitation)


## 3. Leakage audit

Same hunt as Week 3 / the leakage skill, on the **final** ML-08 feature set (`MODEL_NUMERIC_*` + categoricals).

**Checklist (this run):**

| Check | Result |
|---|---|
| Label-derived columns in X? | **No** — `trend_direction`, `trend_pct`, `is_declining_label` blocked; overlap assert passes. |
| Sibling windows that define the label? | **Excluded** — `impressions_last_30d` / `impressions_prev_30d` (and click/session twins) are **not** in `MODEL_NUMERIC_FEATURES`. Deliberately adding them on client-holdout drives Precision@50 → **1.000** (confession). |
| Product / decision flags as features? | Starter CSV has no `health_score` / triage flags in X. Baseline reason codes (`stale_visible_page`, …) stay **outputs**, not inputs. Engineered `measurable_opportunity` is also out of the model list. |
| Future / overlapping windows? | Teaching label is **same-window** (last-30 vs prev-30). 90d aggregates partially overlap that window — disclosed limitation of the starter slice; not a future-outcome design. |
| Attack test (add `trend_pct`) | Client-holdout Precision@50 jumps **0.680 → 1.000** (AP/ROC also ~1.0). Harness works; honest model does **not** include the leak. |

**Verdict:** Final feature set clears the label-sibling and product-flag traps. Remaining honesty limit is the **proxy itself** (same-window trend tag), not a sneaky column in X.


In [3]:
# Leakage attack on the final feature set (client-holdout, same RF)

forbidden = {"trend_direction", "trend_pct", "is_declining_label"}
overlap_clean = forbidden.intersection(feature_columns)
label_siblings = [
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "sessions_last_30d",
    "sessions_prev_30d",
]
siblings_in_model = [c for c in label_siblings if c in MODEL_NUMERIC_FEATURES]
product_like = [
    c
    for c in df.columns
    if any(k in c.lower() for k in ("health", "zombie", "flag", "quick_win", "needs_attention"))
]
product_in_model = [c for c in product_like if c in MODEL_NUMERIC_FEATURES or c in MODEL_CATEGORICAL_FEATURES]

print("=== leakage audit (final feature set) ===")
print(f"forbidden ∩ feature_columns = {overlap_clean or '∅ (clean)'}")
print(f"label-sibling cols in MODEL_NUMERIC_FEATURES = {siblings_in_model or '∅ (excluded)'}")
print(f"product-like cols present in frame = {product_like or '∅'}")
print(f"product-like cols in model lists = {product_in_model or '∅ (not used as features)'}")
print(f"measurable_opportunity in MODEL_NUMERIC? {'measurable_opportunity' in MODEL_NUMERIC_FEATURES}")
print()

# Attack 1: add trend_pct (direct label sibling)
X_leak_pct, cols_pct = build_feature_matrix(df, extra_cols=["trend_pct"])
rf_pct = make_rf()
rf_pct.fit(X_leak_pct.iloc[train_idx], y.iloc[train_idx])
m_pct = ranking_metrics(
    y.iloc[test_idx], rf_pct.predict_proba(X_leak_pct.iloc[test_idx])[:, 1]
)

# Attack 2: add last/prev 30d impressions (label recipe ingredients)
X_leak_win, cols_win = build_feature_matrix(
    df, extra_cols=["impressions_last_30d", "impressions_prev_30d"]
)
rf_win = make_rf()
rf_win.fit(X_leak_win.iloc[train_idx], y.iloc[train_idx])
m_win = ranking_metrics(
    y.iloc[test_idx], rf_win.predict_proba(X_leak_win.iloc[test_idx])[:, 1]
)

honest_p50 = metrics_client["precision_at_50"]
print("--- deliberate leak probes (client-holdout RF) ---")
print(f"honest features          Precision@50 = {honest_p50:.3f}")
print(
    f"+ leak trend_pct          Precision@50 = {m_pct['precision_at_50']:.3f}  "
    f"AP={m_pct['average_precision']:.3f}  ROC-AUC={m_pct['roc_auc']:.3f}"
)
print(
    f"+ leak last/prev 30d imp  Precision@50 = {m_win['precision_at_50']:.3f}  "
    f"AP={m_win['average_precision']:.3f}  ROC-AUC={m_win['roc_auc']:.3f}"
)
print()
print("Collapse toward 1.0 when leaks are added = harness OK; honest run stays at 0.680.")
print("Same-window proxy limit remains: score ≈ 'looks declining in this slice', not future recovery.")


=== leakage audit (final feature set) ===
forbidden ∩ feature_columns = ∅ (clean)
label-sibling cols in MODEL_NUMERIC_FEATURES = ∅ (excluded)
product-like cols present in frame = ∅
product-like cols in model lists = ∅ (not used as features)
measurable_opportunity in MODEL_NUMERIC? False



--- deliberate leak probes (client-holdout RF) ---
honest features          Precision@50 = 0.680
+ leak trend_pct          Precision@50 = 1.000  AP=1.000  ROC-AUC=1.000
+ leak last/prev 30d imp  Precision@50 = 1.000  AP=0.936  ROC-AUC=0.957

Collapse toward 1.0 when leaks are added = harness OK; honest run stays at 0.680.
Same-window proxy limit remains: score ≈ 'looks declining in this slice', not future recovery.


## 4. Claim rewrite

**Boldest sentence I might have wanted (unsafe):**
> "Our Random Forest proves which pages will decline and shows that refreshing them will recover traffic — Precision@50 of 90% beats the hand baseline."

**Problems:** "proves" / "will decline" / "will recover" are causal + future-outcome language the design does not support; **0.900** is the random-split mirage; baseline comparison must use the same honest split.

**Safe rewrite (observed / measured / directional / decision-support):**
> "On the starter teaching slice, a Random Forest **ranked** holdout pages with client-holdout **Precision@50 = 0.680** (34/50 proxy-positives), versus **0.240** for the transparent baseline and a holdout base rate of **0.391**. Under a random row split the same model measured **0.900**, so we treat **0.680** as the honest figure. The label is the same-window proxy `trend_direction == 'down'`, not a future recovery — so the queue is **decision-support for which page to review first**, not evidence that a refresh causes traffic lift."

Words that stay: **observed, measured, ranked, decision-support, proxy, client-holdout.** Words that go: proves, causes, will recover, predicted Google's algorithm.


In [4]:
# Claim receipt: numbers that the safe rewrite is allowed to cite (this run only)

out_dir = Path("work/outputs")
out_dir.mkdir(parents=True, exist_ok=True)

# baseline Precision@50 on the same client-holdout (recompute transparent score)
from ml_utils import normalize, percentile_rank  # noqa: E402


def baseline_refresh_score(frame: pd.DataFrame) -> pd.Series:
    visibility = percentile_rank(np.log1p(frame["impressions_90d"]))
    freshness = percentile_rank(frame["days_since_last_update"])
    position_opp = (
        (1 - normalize(frame["avg_position"].clip(lower=1, upper=50)))
        * visibility
        * (frame["avg_position"] > 0).astype(int)
    )
    depth_gap = (1 - percentile_rank(frame["word_count"])) * visibility
    return (
        0.40 * visibility
        + 0.30 * freshness
        + 0.25 * position_opp
        + 0.05 * depth_gap
    ).clip(0, 1)


base_scores_test = baseline_refresh_score(df).iloc[test_idx].to_numpy()
baseline_holdout = ranking_metrics(y.iloc[test_idx], base_scores_test)
top50_pos = int(
    pd.Series(scores_client, index=y.iloc[test_idx].index)
    .sort_values(ascending=False)
    .index.to_series()
    .pipe(lambda idx: y.loc[idx].head(50).sum())
)

receipt = {
    "assignment": "ML-09",
    "lane": "Refresh / Content Opportunity Scoring",
    "label": "is_declining_label = (trend_direction == 'down')",
    "model": "random_forest",
    "random_state": RANDOM_STATE,
    "split_before": "random_stratified_row",
    "precision_at_50_before_random": metrics_random["precision_at_50"],
    "base_rate_before_random": base_random,
    "split_after": "client_holdout",
    "precision_at_50_after_client": metrics_client["precision_at_50"],
    "base_rate_after_client": base_client,
    "precision_at_50_gap_random_minus_client": gap_p50,
    "baseline_precision_at_50_client_holdout": baseline_holdout["precision_at_50"],
    "holdout_top50_proxy_positives": top50_pos,
    "leak_trend_pct_precision_at_50": m_pct["precision_at_50"],
    "leak_last_prev_30d_precision_at_50": m_win["precision_at_50"],
    "forbidden_overlap": sorted(overlap_clean),
    "paper_findings_audited": [
        "Finding #4 Freshness Multiplier (3.2x health / 57x impressions)",
        "ML appendix RF predicting health_score (circular composite)",
    ],
    "claim_stance": "observed / measured / directional / decision-support",
    "sklearn_version": sklearn.__version__,
}

receipt_path = out_dir / "validation_audit_metrics.json"
receipt_path.write_text(json.dumps(receipt, indent=2) + "\n")

print("=== safe-claim number check ===")
print(f"client-holdout RF Precision@50 = {metrics_client['precision_at_50']:.3f}")
print(f"random-split RF Precision@50   = {metrics_random['precision_at_50']:.3f}")
print(f"client-holdout baseline P@50   = {baseline_holdout['precision_at_50']:.3f}")
print(f"holdout base rate              = {base_client:.3f}")
print(f"top-50 proxy positives         = {top50_pos} / 50")
print(f"Wrote audit receipt: {receipt_path}")
print()
print("SAFE REWRITE (copy-ready):")
print(
    "On the starter teaching slice, a Random Forest ranked holdout pages with "
    f"client-holdout Precision@50 = {metrics_client['precision_at_50']:.3f} "
    f"({top50_pos}/50 proxy-positives), versus "
    f"{baseline_holdout['precision_at_50']:.3f} for the transparent baseline and a "
    f"holdout base rate of {base_client:.3f}. Under a random row split the same model "
    f"measured {metrics_random['precision_at_50']:.3f}, so we treat "
    f"{metrics_client['precision_at_50']:.3f} as the honest figure. The label is the "
    "same-window proxy trend_direction == 'down', not a future recovery — so the queue "
    "is decision-support for which page to review first, not evidence that a refresh "
    "causes traffic lift."
)


=== safe-claim number check ===
client-holdout RF Precision@50 = 0.680
random-split RF Precision@50   = 0.900
client-holdout baseline P@50   = 0.240
holdout base rate              = 0.391
top-50 proxy positives         = 34 / 50
Wrote audit receipt: work/outputs/validation_audit_metrics.json

SAFE REWRITE (copy-ready):
On the starter teaching slice, a Random Forest ranked holdout pages with client-holdout Precision@50 = 0.680 (34/50 proxy-positives), versus 0.240 for the transparent baseline and a holdout base rate of 0.391. Under a random row split the same model measured 0.900, so we treat 0.680 as the honest figure. The label is the same-window proxy trend_direction == 'down', not a future recovery — so the queue is decision-support for which page to review first, not evidence that a refresh causes traffic lift.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
